Other Colabs

1. [Centrality](https://colab.research.google.com/drive/19EfBQSqsmOrdEKVpAoFWprqmGlWXM5pm?usp=sharing)

In [ ]:
# import pandas as pd

# df=pd.read_csv('300_TRANSCRIPT.csv',delimiter='\t')

In [ ]:
# last_speaker=''
# last_sentence=''
# not_printed=True
# sentences = []
# for i,r in df.iterrows():
#     if r['speaker']==last_speaker:
#         last_sentence += ' '+r['value']
#         not_printed = True
#     else:
#         # print(last_sentence)
#         if last_sentence != '':
#             sentences.append(last_sentence)
#         last_sentence = ''
#         last_sentence = r['value']
#         not_printed = False
# if not_printed:
#     # print(last_sentence)
#     sentences.append(last_sentence)


In [ ]:
# # prompt: I have a list of sentences in the variable sentences. I want to remove all stop words from the strings in the list and also lemmatize all sentences.

# import nltk
# nltk.download('stopwords')
# nltk.download('wordnet')
# nltk.download('omw-1.4')
# from nltk.corpus import stopwords
# from nltk.stem import WordNetLemmatizer

# stop_words = set(stopwords.words('english'))
# lemmatizer = WordNetLemmatizer()

# def clean_text(text):
#     words = text.lower().split()
#     words = [lemmatizer.lemmatize(w) for w in words if not w in stop_words]
#     return " ".join(words)

# cleaned_sentences = [clean_text(s) for s in sentences]


In [ ]:
# edges=[]
# for i in range(len(cleaned_sentences)-1):
#   j=i+1
#   for word1 in cleaned_sentences[i].split():
#     for word2 in cleaned_sentences[j].split():
#         edges.append((word1,word2))


In [ ]:
# import networkx as nx
# G=nx.Graph()
# G.add_edges_from(edges)

In [ ]:
# nx.draw_networkx(G)

In [ ]:
# file=open('300_edges.csv','w')
# for edge in edges:
#     file.write(edge[0]+','+edge[1]+'\n')
# file.close()

## Matrix Construction

In [ ]:
# ! unzip daic_woz_cleaned.zip

In [ ]:
import nltk
nltk.download('stopwords')


In [ ]:
import nltk
# nltk.download('punkt')
# nltk.download('wordnet')
# nltk.download('averaged_perceptron_tagger')

import pandas as pd
from nltk.tokenize import MWETokenizer
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
lemmatizer = WordNetLemmatizer()
tokenizer = MWETokenizer()
from itertools import chain
import os
import networkx as nx
import numpy as np
import json 
import sys

from collections import defaultdict

def lemmatize_text_with_pos(text):
    words = text.lower().split()
    tagged_words = nltk.pos_tag(words)
    lemmatized_words = []
    for word, tag in tagged_words:
        if tag.startswith('N'):  # Noun
            pos = 'n'
        elif tag.startswith('V'):  # Verb
            pos = 'v'
        elif tag.startswith('J'):  # Adjective
            pos = 'a'
        elif tag.startswith('R'):  # Adverb
            pos = 'r'
        else:
            pos = 'n'  # Default to noun if unsure
        lemmatized_words.append(lemmatizer.lemmatize(word, pos=pos))
    return lemmatized_words


topk_word_sel = 250
def get_matrix(directory, 
                topk_word_sel = topk_word_sel,
                path_dict_idx_json = f'dict_idx_newshape{topk_word_sel}.json', 
                path_all_word = f'all_word_newshape{topk_word_sel}.txt', 
                path_dir_matrix_out = f'daic_woz_mat_output_newshape{topk_word_sel}', 
                ):
    all_words = set()
    cnt_all_words = defaultdict(lambda : 0)
    stop_words = set(stopwords.words('english'))

    for filename in os.listdir(directory):
        if filename.endswith("_TRANSCRIPT_cleaned.csv"):
            filepath = os.path.join(directory, filename)
            df = pd.read_csv(filepath, delimiter='\t')
            df = df.drop_duplicates()
            df = df.dropna(subset=['value'])

            last_speaker = ''
            last_sentence = ''
            new_rows = []
            for i, row in df.iterrows():
                if row['speaker'] == last_speaker:
                    last_sentence += ' ' + row['value']
                else:
                    if last_sentence != '':
                        new_rows.append([last_speaker, last_sentence])
                    last_sentence = row['value']
                    last_speaker = row['speaker']
            if last_sentence != '':
                new_rows.append([last_speaker, last_sentence])

            new_df = pd.DataFrame(new_rows, columns=['speaker', 'value'])
            new_df['value'] = new_df['value'].str.replace(r'\b(a|an|the)\b', '', regex=True)
            new_df['tokenized_value'] = new_df['value'].apply(lambda x: ' '.join(tokenizer.tokenize(nltk.word_tokenize(x))))
            new_df['lemmatized_value'] = new_df['tokenized_value'].apply(lemmatize_text_with_pos)
            new_df['lemmatized_value'] = new_df['lemmatized_value'].apply(lambda x_list: [x for x in x_list if x.lower() not in stop_words])
            # ! for all test
            # print('lemmatized_value')
            # print(new_df['lemmatized_value'])
            # sys.exit()
            lem_list_list = new_df['lemmatized_value'].to_list()
            for lem_list in lem_list_list:
                for lem in lem_list:
                    cnt_all_words[lem] += 1
            # ! for all test

            all_words.update(chain.from_iterable(new_df['lemmatized_value'].values))
    
    # ! old code
    # all_words = list(sorted_items_sel)
    # indices = {word:i for i,word in enumerate(all_words)}

    # * new code
    sorted_items = sorted(cnt_all_words.items(), key=lambda kv: (kv[1]), reverse=True)
    sorted_items_sel = sorted_items[:topk_word_sel]
    all_words = [key for key, val in sorted_items_sel]
    indices = {word:i for i,word in enumerate(all_words)}

    
    # * save 'all_words' to txt
    with open(path_all_word, 'w') as f:
        for line in all_words:
            f.write(f"{line}\n")
    
    # * save 'indices' to json 
    with open(path_dict_idx_json, "w") as outfile: 
        json.dump(indices, outfile)
    
    matrices=[]
    for filename in os.listdir(directory):
        if filename.endswith("_TRANSCRIPT_cleaned.csv"):
            output_matrix = np.zeros((len(all_words),len(all_words)))
            filepath = os.path.join(directory, filename)
            df = pd.read_csv(filepath, delimiter='\t')
            df = df.drop_duplicates()
            df = df.dropna(subset=['value'])
            last_speaker = ''
            last_sentence = ''
            new_rows = []
            for i, row in df.iterrows():
                if row['speaker'] == last_speaker:
                    last_sentence += ' ' + row['value']
                else:
                    if last_sentence != '':
                        new_rows.append([last_speaker, last_sentence])
                    last_sentence = row['value']
                    last_speaker = row['speaker']
            if last_sentence != '':
                new_rows.append([last_speaker, last_sentence])

            new_df = pd.DataFrame(new_rows, columns=['speaker', 'value'])
            new_df['value'] = new_df['value'].str.replace(r'\b(a|an|the)\b', '', regex=True)
            new_df['tokenized_value'] = new_df['value'].apply(lambda x: ' '.join(tokenizer.tokenize(nltk.word_tokenize(x))))
            new_df['lemmatized_value'] = new_df['tokenized_value'].apply(lemmatize_text_with_pos)
            for i in range(len(new_df['lemmatized_value'])-1):
                for word1 in new_df['lemmatized_value'][i]:
                    if word1 in all_words: 
                        for word2 in new_df['lemmatized_value'][i+1]:
                            if word2 in all_words: 
                                output_matrix[indices[word1],indices[word2]]+=1
            
            if not os.path.exists(path_dir_matrix_out): os.makedirs(path_dir_matrix_out)
            user_id = filename.split('_')[0]
            np.save(os.path.join(path_dir_matrix_out,f'{user_id}_mat'), output_matrix)

            matrices.append(output_matrix)


    return matrices, all_words, indices

matrices,_,_= get_matrix('daic_woz_cleaned/')

In [1]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
# import torchvision
import torchvision.transforms as transforms
import sys

from module_model import LargeImageCNN, CustomImageDataset
from tqdm import tqdm
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


phq_list = ['PHQ8_Binary','PHQ8_Score','PHQ8_NoInterest','PHQ8_Depressed','PHQ8_Sleep',
            'PHQ8_Tired','PHQ8_Appetite','PHQ8_Failure','PHQ8_Concentrating','PHQ8_Moving']
phq_sel = 'PHQ8_Depressed'
assert phq_sel in phq_list
# Hyperparameters
num_epochs = 20
batch_size = 4
learning_rate = 3e-5

# Data loading and preprocessing

# ! set tranformation
# sel_transform = True
sel_transform = False
transform = transforms.Compose([
    transforms.Resize((256, 256)),  # Resize to manageable size for quicker training
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),  # Single-channel mean and std
])

# ! for change dataset
topk_word_sel = 500
# dir_path = os.path.join('daic_woz_mat_output', )
# dir_path = os.path.join('daic_woz_mat_output_newshape1000', )
dir_path = os.path.join(f'daic_woz_mat_output_newshape{topk_word_sel}', )

np_file_list = [x for x in os.listdir(dir_path) if x.endswith('.png') and not x.startswith('.')]
np_file_list = sorted(np_file_list, key=lambda x: int(x.split('_')[0]))
# user_list = 
path_train = os.path.join('label_of_data', 'train_split_Depression_AVEC2017.csv')
path_dev = os.path.join('label_of_data', 'dev_split_Depression_AVEC2017.csv')
# path_test = os.path.join('label_of_data', 'test_split_Depression_AVEC2017_full.csv')
df_train = pd.read_csv(path_train)
df_dev = pd.read_csv(path_dev)


# ! for change dataset
size_multiply = 123008
# model = LargeImageCNN().to(device)
# model = LargeImageCNN(size_multiply = 128).to(device)
model = LargeImageCNN(size_multiply = size_multiply).to(device)

print('pass create model')
# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Training the model
def train_model_one_epoch(train_loader):
    # for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for i, (images_train, labels) in tqdm(enumerate(train_loader)):
        images_train, labels = images_train.to(device), labels.to(device)
        # print('images.shape', images_train.shape)
        # Forward pass
        outputs = model(images_train)
        loss = criterion(outputs, labels)


        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        # if (i + 1) % 2 == 0:
        #     print(f'Epoch [{epoch + 1}/{num_epochs}], Step [{i + 1}/{len(train_loader)}], Loss: {running_loss / 100:.4f}')
        #     running_loss = 0.0
    return running_loss / len(train_loader)

# Test the model
def test_model(dev_loader):
    model.eval()
    correct = 0
    total = 0
    running_loss = 0.0
    with torch.no_grad():
        # for images, labels in test_loader:
        for images, labels in dev_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    # print(f'Accuracy of the model on the 10,000 test images: {100 * correct / total:.2f}%')
    return running_loss / len(dev_loader)



pass create model


In [6]:
import os
num_folds = 5

phq_sel_list = ['PHQ8_NoInterest','PHQ8_Depressed','PHQ8_Sleep',
                'PHQ8_Tired','PHQ8_Appetite','PHQ8_Failure','PHQ8_Concentrating','PHQ8_Moving']
# path_save = os.path.join(os.getcwd(), 'model_save')
path_save = os.path.join(os.getcwd(), f'model_save_top{topk_word_sel}')

if not os.path.exists(path_save) : os.makedirs(path_save)


for phq_sel in phq_sel_list:
    print('=='*50)
    print(f'phq_sel = {phq_sel}')

    user_train = df_train['Participant_ID'].tolist()
    label_train = df_train[phq_sel].tolist()
    label_train = [int(x >=1) for x in label_train]

    user_dev = df_dev['Participant_ID'].tolist()
    label_dev = df_dev[phq_sel].tolist()
    label_dev = [int(x >=1) for x in label_dev]

    np_train_list = [x for x in np_file_list if int(x.split('_')[0]) in user_train]
    np_dev_list = [x for x in np_file_list if int(x.split('_')[0]) in user_dev]

    train_dataset = CustomImageDataset(image_dir = dir_path, 
                                        np_file_list = np_train_list, 
                                        label_list = label_train, 
                                        sel_transform = sel_transform, 
                                        transform = transform)
    dev_dataset = CustomImageDataset(image_dir = dir_path, 
                                        np_file_list = np_dev_list, 
                                        label_list = label_dev, 
                                        sel_transform = sel_transform,
                                        transform = transform)

    # train_dataset = CustomImageDataset

    train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size=batch_size, shuffle=True)
    dev_loader = torch.utils.data.DataLoader(dataset=dev_dataset, batch_size=batch_size, shuffle=False)
    # test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size=batch_size, shuffle=False)

    for fold_idx in range(num_folds):
        best_loss = np.inf
        for epoch in range(num_epochs):
            loss_train = train_model_one_epoch(train_loader)
            loss_dev = test_model(dev_loader)
            print(f'Fold: {fold_idx+1}, Epoch [{epoch + 1}/{num_epochs}], Loss: {loss_train :.4f}, dev_Loss: {loss_dev :.4f}')
            if loss_dev < best_loss:
                best_loss = loss_dev
                print(f'best loss: {best_loss:.4f}')
                # torch.save(model.state_dict(), f'./model_best_{phq_sel}_fold{fold_idx}.pth')
                torch.save(model.state_dict(), os.path.join(path_save,f'model_best_{phq_sel}_top{topk_word_sel}_fold{fold_idx}.pth'))
        # break
    print('=='*50)

    # break

phq_sel = PHQ8_NoInterest


14it [00:04,  3.23it/s]


Fold: 1, Epoch [1/20], Loss: 0.6702, dev_Loss: 0.7195
best loss: 0.7195


14it [00:04,  3.37it/s]


Fold: 1, Epoch [2/20], Loss: 0.6532, dev_Loss: 0.7091
best loss: 0.7091


...

Fold: 5, Epoch [18/20], Loss: 0.0267, dev_Loss: 1.4178


14it [00:03,  3.72it/s]


Fold: 5, Epoch [19/20], Loss: 0.0244, dev_Loss: 1.4701


14it [00:03,  3.68it/s]


Fold: 5, Epoch [20/20], Loss: 0.0244, dev_Loss: 1.4812


In [ ]:
# test_model()
# 128 * 32 * 32
# 32*62*62
32*62*62

In [ ]:
# 64000000 / (32 * 32)

32 * 62 * 62

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


topk_word_sel = 500
# dir_np = os.path.join(os.getcwd(), 'daic_woz_mat_output_newshape500')
# dir_np = os.path.join(os.getcwd(), 'daic_woz_mat_output')
# dir_np = os.path.join(os.getcwd(), 'daic_woz_mat_output_newshape1000')
# dir_np = os.path.join(os.getcwd(), 'daic_woz_mat_output_newshape500')
dir_np = os.path.join(os.getcwd(), f'daic_woz_mat_output_newshape{topk_word_sel}')

file_npy = [x for x in os.listdir(dir_np) if x.endswith('.npy') and not x.startswith('.')]
file_npy = sorted(file_npy, key=lambda x : int(x.split('_')[0]))
path_list = [os.path.join(dir_np,x) for x in file_npy]
max_all = 0 

for path_file in path_list:
    np_load = np.load(path_file)
    if np.max(np_load) > max_all:
        max_all = np.max(np_load)
# max_all = 416

# rescale 

from imageio import imwrite
from multiprocessing import Pool
from pathlib import Path
from PIL import Image

rootPath = Path('daic_woz_mat_output')

def npy2png(npyFile, max_scale = max_all):
    np_load = np.load(npyFile) * 255 / max_scale
    np_load = np_load.astype(np.uint8)
    image = Image.fromarray(np_load)
    npyFile_png = npyFile[:-len('.npy')] + '.png'
    image.save(npyFile_png)

for path_file in path_list:
    npy2png(path_file, max_scale = max_all)

In [ ]:
max_all

## model eval

In [11]:
import os, sys
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt

from module_model import LargeImageCNN, CustomImageDataset
import torch
import torch.nn as nn
import torchvision.transforms as transforms

# import sklearn
from sklearn.metrics import (
    precision_recall_fscore_support,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    )
# roc_auc_score(y, clf.predict_proba(X)[:, 1])
# 'binary'
# confusion_matrix(y_true, y_pred)

batch_size = 8
transform = transforms.Compose([
    transforms.Resize((256, 256)),  # Resize to manageable size for quicker training
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),  # Single-channel mean and std
])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ! sel topk_word_sel 
topk_word_sel = 500
# dir_path = os.path.join('daic_woz_mat_output', )
dir_path = os.path.join(f'daic_woz_mat_output_newshape{topk_word_sel}', )


np_file_list = [x for x in os.listdir(dir_path) if x.endswith('.png') and not x.startswith('.')]
np_file_list = sorted(np_file_list, key=lambda x: int(x.split('_')[0]))
# user_list = 
path_train = os.path.join('label_of_data', 'train_split_Depression_AVEC2017.csv')
path_dev = os.path.join('label_of_data', 'dev_split_Depression_AVEC2017.csv')
# path_test = os.path.join('label_of_data', 'test_split_Depression_AVEC2017_full.csv')
df_train = pd.read_csv(path_train)
df_dev = pd.read_csv(path_dev)

# dir_model_save = os.path.join(os.getcwd(), 'model_save')
dir_model_save = os.path.join(os.getcwd(), f'model_save_top{topk_word_sel}')

phq_sel_list = ['PHQ8_NoInterest','PHQ8_Depressed','PHQ8_Sleep',
                'PHQ8_Tired','PHQ8_Appetite','PHQ8_Failure',
                'PHQ8_Concentrating','PHQ8_Moving']

for phq_sel in phq_sel_list:
    # file_model_list = [x for x in os.listdir(dir_model_save) 
    #                     if x.endswith('.pth') and x.startswith(f'model_best_{phq_sel}') and 
    #                     f'top{topk_word_sel}' in x]
    file_model_list = [x for x in os.listdir(dir_model_save) 
                        if x.endswith('.pth') and 
                        x.startswith(f'model_best_{phq_sel}') and 
                        f'top{topk_word_sel}' in x]
    # print(file_model_list)
    # sys.exit()
    file_model_list = sorted(file_model_list)
    path_list = [os.path.join(dir_model_save, x) for x in file_model_list]

    softmax_layer = nn.Softmax(dim=1)
    # * load model from save state

    user_train = df_train['Participant_ID'].tolist()
    label_train = df_train[phq_sel].tolist()
    label_train = [int(x >=1) for x in label_train]

    # user_dev = df_dev['Participant_ID'].tolist()
    # label_dev = df_dev[phq_sel].tolist()
    # label_dev = [int(x >=1) for x in label_dev]
    np_dev_list = [x for x in np_file_list if int(x.split('_')[0]) in user_dev]

    # plt.hist(label_train)
    # # plt.hist(label_dev)
    # plt.title(f'{phq_sel}')
    # plt.show()
    dev_dataset = CustomImageDataset(image_dir = dir_path, 
                                        np_file_list = np_dev_list, 
                                        label_list = label_dev, 
                                        sel_transform = sel_transform,
                                        transform = transform)
    dev_loader = torch.utils.data.DataLoader(dataset=dev_dataset, batch_size=batch_size, shuffle=False)

    list_text_pr = []
    for idx_fold, path_sel in enumerate(path_list):
        

        # model = LargeImageCNN(size_multiply = 246016).to(device)
        list_text_pr.append('=='*50)
        list_text_pr.append(f'phq_sel = {phq_sel}, fold : {idx_fold}')
        model = LargeImageCNN(size_multiply=size_multiply)
        model.load_state_dict(torch.load(path_sel, weights_only=True))
        model.eval()

        with torch.no_grad():
            # for images, labels in test_loader:
            pred_class_list = []
            pred_prob_list = []
            labels_list = []
            for images, labels in dev_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                outputs_prob = softmax_layer(outputs)
                class_1_prob = outputs_prob[:, 1]

                pred_class_list += list(predicted.numpy(force=True))
                pred_prob_list += list(class_1_prob.numpy(force=True))
                labels_list += list(labels.numpy(force=True))
                # break

            # print('pred_class_list' , pred_class_list)
            # print('pred_prob_list' , pred_prob_list)
            # print('labels_list' , labels_list)
            # sys.exit()

            # print(f'len(pred_class_list) = {len(pred_class_list)}')
            # print(f'len(pred_prob_list) = {len(pred_prob_list)}')
            # print(f'len(labels_list) = {len(labels_list)}')

            # print(f'len(pred_class_list) = {pred_class_list}')
            # print(f'len(pred_prob_list) = {pred_prob_list}')
            # print(f'len(labels_list) = {labels_list}')
            # sys.exit()

            # prec, recall, f1 = precision_recall_fscore_support(
            #                     labels_list, pred_class_list, average='binary')
            roc_auc = roc_auc_score(labels_list, pred_prob_list)
            # conf_mat = confusion_matrix(labels_list, pred_prob_list, labels=[0, 1])
            list_text_pr.append(classification_report(labels_list, pred_class_list, labels=[0, 1]))
            list_text_pr.append(f'roc_auc = {roc_auc}')
            # print(f'conf_mat = {conf_mat}')
            list_text_pr.append('=='*50)
    
    with open(f'phq_sel_{phq_sel}_top{topk_word_sel}_transcript.txt', "w") as text_file:
        text_file.write('\n'.join(list_text_pr))


In [ ]:

phq_sel_list

In [ ]:
# import nltk
# from nltk.corpus import stopwords

# # Download stopwords list if you haven't already
# nltk.download('stopwords')

# # Define your list of words
# words = ["This", "is", "a", "simple", "example", "to", "remove", "stopwords"]

# # Get the list of stopwords in English
# stop_words = set(stopwords.words('english'))

# # Filter out stopwords
# filtered_words = [word for word in words if word.lower() not in stop_words]

# print(filtered_words)